In [ ]:
%pip install tensorflow
%pip install keras
%pip install scikit-learn

In [49]:
import pandas as pd
import numpy as np
import re
import pickle

from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout # type: ignore
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
import tensorflow.keras.preprocessing.sequence # type: ignore
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [50]:
# loading data
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

encoder = LabelEncoder()
df['label_num'] = encoder.fit_transform(df['label'])


In [51]:
#cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['clean_message'] = df['message'].apply(clean_text)


In [52]:
# token
vocab_size = 10000  
max_length = 150    
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"   

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(df['clean_message'])

sequences = tokenizer.texts_to_sequences(df['clean_message'])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [53]:
#cnn model and training
embedding_dim = 64

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    
    Conv1D(128, 5, activation='relu'),
    
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.5), 
    
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

#training
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df['label_num'], test_size=0.2, random_state=42)

history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

model.save('cnn_spam_model.keras')


c:\Users\maro\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_3          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.8759 - loss: 0.3282 - val_accuracy: 0.9704 - val_loss: 0.1108
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9735 - loss: 0.0708 - val_accuracy: 0.9857 - val_loss: 0.0644
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9960 - loss: 0.0217 - val_accuracy: 0.9848 - val_loss: 0.0645
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 36ms/step - accuracy: 0.9989 - loss: 0.0121 - val_accuracy: 0.9839 - val_loss: 0.0755
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.9996 - loss: 0.0094 - val_accuracy: 0.9839 - val_loss: 0.0744


In [ ]:
def predict_spam_cnn(input_text):
    """Function to be called by the GUI to get a prediction."""
    cleaned_text = clean_text(input_text)
    
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding=padding_type, truncating=trunc_type)
    
    prediction_prob = model.predict(padded_sequence, verbose=0)[0][0]
    
    if prediction_prob >= 0.5:
        return "Spam"
    else:
        return "Not Spam"

test_msg = "hey my name is Tristan and you just won a free ticket to the Bahamas! Click here to claim now!"
print(f"\nTest Prediction for: '{test_msg}'")
print(f"Result: {predict_spam_cnn(test_msg)}")


Test Prediction for: 'hey my name is maro and you just won a free ticket to the Bahamas! Click here to claim now!'
Result: Spam
